In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from xgboost import XGBRegressor

In [2]:
train = pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv")
test = pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv")

print(train.shape)
print(test.shape)

train.head()

(1460, 81)
(1459, 80)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [3]:
y = train["SalePrice"]

train_id = train["Id"]
test_id = test["Id"]

train = train.drop(["Id","SalePrice"], axis=1)
test = test.drop(["Id"], axis=1)

In [4]:
data = pd.concat([train, test])
print(data.shape)

(2919, 79)


In [5]:
# Fill numeric values
for col in data.select_dtypes(include=["float64","int64"]):
    data[col] = data[col].fillna(data[col].median())

# Fill categorical values
for col in data.select_dtypes(include=["object"]):
    data[col] = data[col].fillna("None")

In [6]:
for col in data.select_dtypes(include=["object"]):
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))

In [7]:
X_train = data[:len(train)]
X_test = data[len(train):]

y = np.log1p(y)

model = XGBRegressor(
    n_estimators=3000,
    learning_rate=0.01,
    max_depth=4
)

model.fit(X_train, y)

pred = model.predict(X_test)
pred = np.expm1(pred)

In [8]:
submission = pd.DataFrame({
    "Id": test_id,
    "SalePrice": pred
})

submission.to_csv("submission.csv", index=False)

submission.head()

,Id,SalePrice
0,1461,120829.109375
1,1462,153958.609375
2,1463,183032.250000
3,1464,188881.593750
4,1465,179653.546875
